# NexoraLM: Educational GPT-Style Language Model Training on Google Colab

Welcome to **NexoraLM**! This notebook executes full end-to-end training of a GPT-style decoder-only Transformer entirely from scratch using PyTorch and GPU acceleration.

### Overview
- **Architecture**: Decoder-Only Transformer (Pre-LN, Causal Scaled Dot-Product Attention, GELU FFN, Weight Tying).
- **Tokenizer**: Custom character-level tokenizer (69 tokens) with `<pad>`, `<unk>`, `<bos>`, `<eos>`.
- **Dataset**: TinyShakespeare public domain corpus (1.11MB, 90% train / 10% val).
- **Checkpoints & Outputs**: Automatically persisted to your Google Drive to prevent loss upon runtime disconnection.


## 1. Mount Google Drive
Mount Google Drive to persist trained checkpoints, training metrics, and loss plots across runtime restarts.


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Create a dedicated directory on Google Drive for NexoraLM
GDRIVE_DIR = '/content/drive/MyDrive/NexoraLM_Colab'
os.makedirs(f'{GDRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{GDRIVE_DIR}/outputs', exist_ok=True)
print(f'Google Drive directory ready at: {GDRIVE_DIR}')


## 2. Hardware Accelerator & Environment Verification
Detect GPU hardware, CUDA capability, and memory budget.


In [ ]:
import torch
import sys

print(f'Python Version : {sys.version.split()[0]}')
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    device = 'cuda'
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    cuda_ver = torch.version.cuda
    print(f'GPU Device     : {gpu_name}')
    print(f'Total VRAM     : {gpu_mem:.2f} GB')
    print(f'CUDA Version   : {cuda_ver}')
else:
    device = 'cpu'
    print('WARNING: CUDA GPU not detected. Falling back to CPU mode.')


## 3. Install Dependencies
Install required packages for the NexoraLM pipeline.


In [ ]:
!pip install --quiet matplotlib pandas tqdm pytest


## 4. Load NexoraLM Source Code
Choose one of the following methods to load the NexoraLM codebase into `/content/NexoraLM`:

- **Option A (Recommended)**: Clone your GitHub repository (enter your repo URL below).
- **Option B**: Upload a zipped project archive (`NexoraLM.zip`).


In [ ]:
import os
import shutil

# Option A: Clone from GitHub repository
GITHUB_REPO_URL = ''  # e.g., 'https://github.com/yourusername/NexoraLM.git'

if GITHUB_REPO_URL.strip():
    if os.path.exists('/content/NexoraLM'):
        !rm -rf /content/NexoraLM
    !git clone {GITHUB_REPO_URL} /content/NexoraLM
    %cd /content/NexoraLM
else:
    print('Option B: If you do not have a Git remote, upload NexoraLM.zip to /content and uncomment below:')
    # if os.path.exists('/content/NexoraLM.zip'):
    #     !unzip -q /content/NexoraLM.zip -d /content/
    #     %cd /content/NexoraLM
    if os.path.exists('/content/NexoraLM'):
        %cd /content/NexoraLM
    else:
        print('Current working directory:', os.getcwd())


## 5. Dataset and Tokenizer Verification
Verify that the TinyShakespeare dataset files and vocabulary mapping are present and correctly formatted.


In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())

from tokenizer.tokenizer import CharacterTokenizer
from data.preprocessing import load_text_file

vocab_file = 'tokenizer/vocabulary.json'
train_file = 'data/processed/train.txt'
val_file = 'data/processed/val.txt'

assert os.path.exists(vocab_file), f'Missing {vocab_file}'
assert os.path.exists(train_file), f'Missing {train_file}'
assert os.path.exists(val_file), f'Missing {val_file}'

tokenizer = CharacterTokenizer()
tokenizer.load_vocabulary(vocab_file)
train_text = load_text_file(train_file)
val_text = load_text_file(val_file)

print(f'Vocabulary Size : {tokenizer.vocab_size} tokens')
print(f'Training Corpus : {len(train_text):,} characters')
print(f'Val Corpus      : {len(val_text):,} characters')
print('Sample Roundtrip Test:')
sample = 'To be, or not to be.'
assert tokenizer.decode(tokenizer.encode(sample)) == sample
print('Tokenizer verified successfully!')


## 6. Model Construction & Parameter Inspection
Build the `GPTLanguageModel` with dynamic vocabulary binding and print the total parameter count.


In [ ]:
from config.config import ModelConfig
from model.gpt_model import GPTLanguageModel

# Model Configuration
model_config = ModelConfig.from_tokenizer(
    tokenizer=tokenizer,
    embed_dim=256,
    num_layers=4,
    num_heads=4,
    context_length=256,
    dropout=0.1,
    tie_weights=True
)

model = GPTLanguageModel(model_config)
total_params = model.get_num_params()
print(f'Model Architecture Initialized:')
print(f'  - Embedding Dimension : {model_config.embed_dim}')
print(f'  - Transformer Layers  : {model_config.num_layers}')
print(f'  - Attention Heads     : {model_config.num_heads} (head_dim: {model_config.head_dim})')
print(f'  - Context Length      : {model_config.context_length}')
print(f'  - Vocabulary Size     : {model_config.vocab_size}')
print(f'  - Total Parameters    : {total_params:,} (~{total_params/1e6:.2f}M)')


## 7. GPU Memory Verification & Safe Batch Sizing
Check available memory and configure batch size appropriately to prevent Out-Of-Memory (OOM) errors.


In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    allocated = torch.cuda.memory_allocated(0) / (1024**2)
    reserved = torch.cuda.memory_reserved(0) / (1024**2)
    print(f'Allocated VRAM: {allocated:.1f} MB | Reserved VRAM: {reserved:.1f} MB')
    
    # On NVIDIA T4 (15-16 GB) or similar GPUs, batch size 32 or 64 is safe and fast
    batch_size = 32
else:
    batch_size = 8

print(f'Selected Batch Size: {batch_size}')


## 8. Prepare Datasets & DataLoaders
Transform text into input and target next-token sequence pairs.


In [ ]:
from data.dataset import TextDataset, create_dataloader

train_tokens = tokenizer.encode(train_text)
val_tokens = tokenizer.encode(val_text)

train_dataset = TextDataset(train_tokens, context_length=model_config.context_length)
val_dataset = TextDataset(val_tokens, context_length=model_config.context_length)

train_loader = create_dataloader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = create_dataloader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=False)

print(f'Train Batches: {len(train_loader):,} | Validation Batches: {len(val_loader):,}')


## 9. Training Configuration & Mixed Precision (AMP)
Configure the optimizer, learning rate scheduler, and mixed-precision settings.


In [ ]:
from config.config import TrainingConfig
from training.trainer import Trainer
from training.checkpoint import CheckpointManager
from training.metrics import MetricsTracker

# Training Configuration pointing to Google Drive
training_config = TrainingConfig(
    batch_size=batch_size,
    learning_rate=3e-4,
    min_lr=3e-5,
    weight_decay=0.01,
    max_epochs=10,
    warmup_steps=100,
    grad_clip=1.0,
    checkpoint_dir=f'{GDRIVE_DIR}/checkpoints',
    output_dir=f'{GDRIVE_DIR}/outputs',
    device='cuda' if torch.cuda.is_available() else 'cpu',
    use_amp=torch.cuda.is_available()
)

print(f'Training Target Device: {training_config.device}')
print(f'Automatic Mixed Precision (AMP): {training_config.use_amp}')


## 10. Quick Smoke Test Run
Run a small 1-epoch verification on a small batch slice to confirm gradients, forward/backward passes, and checkpoint saving.


In [ ]:
# Smoke test setup with small batch limits
smoke_model = GPTLanguageModel(model_config)
smoke_trainer = Trainer(
    model=smoke_model,
    train_loader=train_loader,
    val_loader=val_loader,
    training_config=TrainingConfig(
        max_epochs=1,
        eval_iters=5,
        device=training_config.device,
        use_amp=training_config.use_amp,
        checkpoint_dir=f'{GDRIVE_DIR}/checkpoints',
        output_dir=f'{GDRIVE_DIR}/outputs'
    ),
    model_config=model_config
)

print('Executing 1-epoch smoke test...')
smoke_loss = smoke_trainer.train_epoch(1)
smoke_val_loss = smoke_trainer.evaluate(max_iters=5)
print(f'Smoke Test Complete! Train Loss: {smoke_loss:.4f} | Val Loss: {smoke_val_loss:.4f}')


## 11. Full GPU Training Execution
Execute full training with loss tracking, learning rate warmup/decay, and checkpoint saving to Google Drive.


In [ ]:
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    training_config=training_config,
    model_config=model_config,
    checkpoint_manager=CheckpointManager(training_config.checkpoint_dir),
    metrics_tracker=MetricsTracker(training_config.output_dir)
)

# Start multi-epoch training
summary = trainer.train()
print('Full training run complete!')
print(f'Best Validation Loss: {summary["best_val_loss"]:.4f}')


## 12. Resume Training After Interruption
If your Colab session disconnected or timed out, run this cell to seamlessly resume training from the latest checkpoint on Google Drive.


In [ ]:
latest_ckpt_path = f'{GDRIVE_DIR}/checkpoints/latest.pt'

if os.path.exists(latest_ckpt_path):
    print(f'Found existing checkpoint at {latest_ckpt_path}. Resuming...')
    resumed_model = GPTLanguageModel(model_config)
    resumed_trainer = Trainer(
        model=resumed_model,
        train_loader=train_loader,
        val_loader=val_loader,
        training_config=training_config,
        model_config=model_config
    )
    ckpt = resumed_trainer.checkpoint_manager.load_checkpoint(
        checkpoint_path=latest_ckpt_path,
        model=resumed_trainer.model,
        optimizer=resumed_trainer.optimizer,
        scheduler=resumed_trainer.scheduler,
        device=training_config.device
    )
    resumed_trainer.start_epoch = ckpt['epoch'] + 1
    resumed_trainer.global_step = ckpt['step']
    resumed_trainer.best_val_loss = ckpt['best_val_loss']
    print(f'Successfully resumed from epoch {ckpt["epoch"]}, step {ckpt["step"]}.')
    # Uncomment to continue training:
    # resumed_trainer.train()
else:
    print('No checkpoint found to resume.')


## 13. Training & Validation Loss Visualization
Display the loss curve generated during training.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

plot_file = f'{GDRIVE_DIR}/outputs/loss_curve.png'
if os.path.exists(plot_file):
    img = mpimg.imread(plot_file)
    plt.figure(figsize=(10, 6), dpi=120)
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print('Loss curve image not found. Ensure training completed at least one epoch.')


## 14. Minimal Text Generation Preview (Sanity Check)
Minimal autoregressive generation routine to visually inspect qualitative model outputs.
*(Full sampling and inference pipeline will be developed in Phase 7).*


In [ ]:
import torch.nn.functional as F

@torch.no_grad()
def minimal_generate(model, tokenizer, prompt, max_new_tokens=150, temperature=0.8, device='cuda'):
    model.eval()
    tokens = tokenizer.encode(prompt)
    x = torch.tensor([tokens], dtype=torch.long, device=device)
    
    for _ in range(max_new_tokens):
        # Crop context if length exceeds context_length
        x_cond = x[:, -model_config.context_length:]
        logits, _ = model(x_cond)
        # Take logits at final position and scale by temperature
        next_token_logits = logits[:, -1, :] / max(1e-5, temperature)
        probs = F.softmax(next_token_logits, dim=-1)
        # Sample next token
        next_token = torch.multinomial(probs, num_samples=1)
        x = torch.cat((x, next_token), dim=1)
        
    return tokenizer.decode(x[0].tolist())

test_prompt = 'KING RICHARD:
'
print('--- Generated Sample ---')
print(minimal_generate(model, tokenizer, test_prompt, max_new_tokens=150, temperature=0.8, device=training_config.device))


## 15. Export Final Model & Artifacts
Copy the best model checkpoint, model configuration, and tokenizer vocabulary into a standalone export bundle.


In [ ]:
import shutil, json

EXPORT_DIR = f'{GDRIVE_DIR}/export_nexoralm'
os.makedirs(EXPORT_DIR, exist_ok=True)

# Copy best model checkpoint
shutil.copyfile(f'{GDRIVE_DIR}/checkpoints/best.pt', f'{EXPORT_DIR}/best.pt')

# Copy vocabulary mapping
shutil.copyfile('tokenizer/vocabulary.json', f'{EXPORT_DIR}/vocabulary.json')

# Save model config metadata
with open(f'{EXPORT_DIR}/model_config.json', 'w') as f:
    json.dump(vars(model_config), f, indent=2)

print(f'Model successfully exported to: {EXPORT_DIR}')
print('Contents:', os.listdir(EXPORT_DIR))
